In [15]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get('https://builtin.com/jobs?search=ai+engineer&country=USA&allLocations=true')

time.sleep(3)

all_soup = BeautifulSoup(driver.page_source, 'lxml')
driver.quit()

jobs = all_soup.find_all('div', {'data-id': 'job-card'})

for job in jobs:
    job_id = job['id'].replace('job-card-', '')
    print(job_id)

8942381
7786866
8668169
8668175
8668163
8668173
8579364
8578460
8348181
8579467
8577973
9063886
9062130
8384879
8237390
7099527
5165505
8827782
8823061
8829957
8819328
8829939
8917070
8327260
8822328


In [16]:
import requests
from bs4 import BeautifulSoup
import json
import time

all_jobs = []

for job in jobs:
    job_id = job['id'].replace('job-card-', '')
    
    # get the real url from the card
    job_link = job.find('a', {'data-id': 'job-card-title'})
    
    if job_link is None:
        print(f'✗ {job_id} - no link found, skipping')
        continue
    
    job_url = job_link['href']
    print(f'fetching: https://builtin.com{job_url}')
    
    page = requests.get(f'https://builtin.com{job_url}')
    soup = BeautifulSoup(page.content, 'lxml')
    
    script = soup.find('script', {'type': 'application/ld+json'})
    
    if script is None:
        print(f'✗ {job_id} - no data, skipping')
        continue
    
    data = json.loads(script.string)
    
    for item in data['@graph']:
        if item.get('@type') == 'JobPosting':
            all_jobs.append(item)
            print(f'✓ {job_id} - {item.get("title")}')
    
    time.sleep(1)

print(f'\ntotal jobs: {len(all_jobs)}')

fetching: https://builtin.com/job/ai-engineer/8942381
✓ 8942381 - AI Engineer
fetching: https://builtin.com/job/technical-lead/7786866
✓ 7786866 - AI Engineer
fetching: https://builtin.com/job/principal-agentic-ai-engineer/8668169
✓ 8668169 - Principal Agentic AI Engineer
fetching: https://builtin.com/job/senior-agentic-ai-engineer/8668175
✓ 8668175 - Senior Agentic AI Engineer
fetching: https://builtin.com/job/principal-agentic-ai-engineer/8668163
✓ 8668163 - Principal Agentic AI Engineer
fetching: https://builtin.com/job/senior-agentic-ai-engineer/8668173
✓ 8668173 - Senior Agentic AI Engineer
fetching: https://builtin.com/job/applied-ai-engineer/8579364
✓ 8579364 - Applied AI Engineer
fetching: https://builtin.com/job/ai-engineer/8578460
✓ 8578460 - AI Engineer
fetching: https://builtin.com/job/senior-enterprise-ai-engineer/8348181
✓ 8348181 - Senior Enterprise AI Engineer
fetching: https://builtin.com/job/staff-ai-engineer/8579467
✓ 8579467 - Staff AI Engineer
fetching: https://bui

In [17]:
print(all_jobs[0].keys())

dict_keys(['EducationRequirements', '@type', 'title', 'description', 'directApply', 'identifier', 'baseSalary', 'datePosted', 'employmentType', 'experienceRequirements', 'applicantLocationRequirements', 'hiringOrganization', 'industry', 'jobBenefits', 'jobLocation', 'validThrough'])


In [18]:
import csv
import datetime

def extract(job):
    return {
        'title'             : job.get('title'),
        'industry'          : ', '.join(job.get('industry', [])),
        'employment_type'   : job.get('employmentType'),
        'experience_months' : job.get('experienceRequirements', {}).get('monthsOfExperience'),
        'education'         : job.get('EducationRequirements', {}).get('credentialCategory'),
        'description'       : job.get('description'),
        'date_posted'       : job.get('datePosted'),
        'scraped_at'        : datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    }

flat_jobs = [extract(job) for job in all_jobs]

with open('jobs.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=flat_jobs[0].keys())
    writer.writeheader()
    writer.writerows(flat_jobs)

print(f'saved {len(flat_jobs)} jobs to jobs.csv')

saved 25 jobs to jobs.csv
